# 🔥 Orman Yangını Erken Uyarı Sistemi
**Model:** EfficientNet-B0 (Transfer Learning) | **Framework:** PyTorch

```
dataset/
├── train/
│   ├── fire/      ← 200 jpg
│   └── no_fire/   ← 200 jpg
└── test/
    ├── fire/      ← 50 jpg
    └── no_fire/   ← 50 jpg
```

## 📦 1 — Kurulum

In [ ]:
!pip install torch torchvision timm gradio scikit-learn matplotlib seaborn tqdm pillow -q

In [ ]:
import os, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder
import timm

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# ── Seed ──────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ── Cihaz ─────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Cihaz : {device}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## ⚙️ 2 — Ayarlar

In [ ]:
# ── Yollar ────────────────────────────────────────────────
TRAIN_DIR   = './dataset/train'   # fire/ ve no_fire/ klasörleri burada
TEST_DIR    = './dataset/test'
WEIGHTS_OUT = './model_weights.pth'

# ── Hiperparametreler ─────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 16    # az veri, 16 yeterli
NUM_EPOCHS  = 20    # 200 görüntü için ideal
LR          = 1e-3
VAL_RATIO   = 0.15  # train'in %15'i validation

print('✅ Ayarlar hazır')

## 🗂️ 3 — Veri Yükleme & Augmentation

In [ ]:
# Train: augmentation açık → az veriyi çoğaltır
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],   # ImageNet mean
                         [0.229, 0.224, 0.225])    # ImageNet std
])

# Val/Test: augmentation kapalı
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ── Dataset yükle ─────────────────────────────────────────
full_train = ImageFolder(TRAIN_DIR, transform=train_transform)
test_ds    = ImageFolder(TEST_DIR,  transform=eval_transform)

# Train / Val split
n_val   = int(len(full_train) * VAL_RATIO)
n_train = len(full_train) - n_val
train_ds, val_ds = random_split(full_train, [n_train, n_val],
                                generator=torch.Generator().manual_seed(SEED))

# Val için eval transform uygula
val_ds.dataset = ImageFolder(TRAIN_DIR, transform=eval_transform)

# ── DataLoader ────────────────────────────────────────────
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

CLASS_NAMES = full_train.classes   # ['fire', 'no_fire']
print(f'Sınıflar    : {CLASS_NAMES}')
print(f'Train       : {n_train} görüntü')
print(f'Validation  : {n_val} görüntü')
print(f'Test        : {len(test_ds)} görüntü')

In [ ]:
# Örnek görüntüleri göster
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    return torch.clamp(tensor * std + mean, 0, 1)

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i >= len(imgs): break
    img = denormalize(imgs[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[labels[i]], fontsize=12,
                 color='red' if CLASS_NAMES[labels[i]]=='fire' else 'green')
    ax.axis('off')
plt.suptitle('Örnek Eğitim Görüntüleri (Augmented)', fontsize=14)
plt.tight_layout()
plt.show()

## 🧠 4 — Model: EfficientNet-B0 (Transfer Learning)

In [ ]:
# ── EfficientNet-B0 yükle (ImageNet ağırlıkları ile) ──────
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=2)

# İlk aşama: sadece son katmanı eğit (backbone donduruldu)
for name, param in model.named_parameters():
    if 'classifier' not in name:
        param.requires_grad = False

model = model.to(device)

# Eğitilecek parametre sayısı
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Toplam parametre    : {total:,}')
print(f'Eğitilecek parametre: {trainable:,} ({100*trainable/total:.1f}%)')
print('\n✅ Model hazır — Phase 1: Sadece classifier eğitilecek')

## 🏋️ 5 — Eğitim

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct = 0.0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct = 0.0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out  = model(imgs)
        loss = criterion(out, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

print('✅ Fonksiyonlar hazır')

In [ ]:
# ── PHASE 1: Backbone dondurulmuş, sadece classifier ──────
print('='*55)
print('PHASE 1 — Classifier eğitimi (5 epoch)')
print('='*55)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
best_val_acc = 0.0

for epoch in range(5):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer)
    vl_loss, vl_acc = evaluate(model, val_loader)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    flag = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), WEIGHTS_OUT)
        flag = '  ✅ kaydedildi'

    print(f'Epoch {epoch+1:2d}/5  |  '
          f'Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.3f}  |  '
          f'Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.3f}{flag}')

print(f'\nPhase 1 tamamlandı. En iyi val acc: {best_val_acc:.3f}')

In [ ]:
# ── PHASE 2: Tüm model açılır, düşük LR ile fine-tune ─────
print('='*55)
print('PHASE 2 — Fine-tuning (tüm model, düşük LR)')
print('='*55)

for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=1e-4)  # çok daha düşük LR!
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS-5)

for epoch in range(NUM_EPOCHS - 5):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer)
    vl_loss, vl_acc = evaluate(model, val_loader)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    flag = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), WEIGHTS_OUT)
        flag = '  ✅ kaydedildi'

    print(f'Epoch {epoch+6:2d}/{NUM_EPOCHS}  |  '
          f'Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.3f}  |  '
          f'Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.3f}{flag}')

print(f'\n🎉 Eğitim tamamlandı! En iyi val acc: {best_val_acc:.3f}')
print(f'💾 Model kaydedildi: {WEIGHTS_OUT}')

In [ ]:
# Eğitim grafiği
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss')
axes[0].plot(epochs, history['val_loss'],   'r-o', label='Val Loss')
axes[0].axvline(x=5.5, color='gray', linestyle='--', label='Fine-tune başlangıcı')
axes[0].set_title('Loss', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs, history['train_acc'], 'b-o', label='Train Acc')
axes[1].plot(epochs, history['val_acc'],   'r-o', label='Val Acc')
axes[1].axvline(x=5.5, color='gray', linestyle='--', label='Fine-tune başlangıcı')
axes[1].set_title('Accuracy', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.suptitle('Eğitim Geçmişi', fontsize=15)
plt.tight_layout()
plt.savefig('./training_history.png', dpi=150)
plt.show()

## 📊 6 — Test & Değerlendirme

In [ ]:
# En iyi modeli yükle
model.load_state_dict(torch.load(WEIGHTS_OUT, map_location=device))
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out  = model(imgs)
        probs = torch.softmax(out, dim=1)
        preds = out.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

print('─'*45)
print('TEST SONUÇLARI')
print('─'*45)
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Test Seti', fontsize=13)
plt.tight_layout()
plt.savefig('./confusion_matrix.png', dpi=150)
plt.show()

# Özet metrikler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
print(f'Accuracy  : {accuracy_score(all_labels, all_preds):.4f}')
print(f'F1 Score  : {f1_score(all_labels, all_preds, average="weighted"):.4f}')
print(f'Precision : {precision_score(all_labels, all_preds, average="weighted"):.4f}')
print(f'Recall    : {recall_score(all_labels, all_preds, average="weighted"):.4f}')

In [ ]:
# Yanlış tahminleri görselleştir
test_ds_raw = ImageFolder(TEST_DIR, transform=eval_transform)
wrong_indices = [i for i, (p, l) in enumerate(zip(all_preds, all_labels)) if p != l]

print(f'Toplam yanlış tahmin: {len(wrong_indices)} / {len(all_labels)}')

if wrong_indices:
    show_n = min(8, len(wrong_indices))
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, idx in zip(axes.flat, wrong_indices[:show_n]):
        img_tensor, true_label = test_ds_raw[idx]
        img = denormalize(img_tensor).permute(1, 2, 0).numpy()
        pred_label = all_preds[idx]
        conf = all_probs[idx][pred_label]
        ax.imshow(img)
        ax.set_title(f'Gerçek: {CLASS_NAMES[true_label]}\nTahmin: {CLASS_NAMES[pred_label]} ({conf:.2f})',
                     color='red', fontsize=9)
        ax.axis('off')
    plt.suptitle('❌ Yanlış Tahminler', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('🎉 Hiç yanlış tahmin yok!')

## 🌐 7 — Gradio Demo (Osman'ın web sitesi için de kullanılabilir)

In [ ]:
import gradio as gr

# Model hazır (zaten yüklü)
model.eval()

def predict(image):
    """Gradio'dan gelen PIL image → tahmin"""
    img = eval_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        out   = model(img)
        probs = torch.softmax(out, dim=1)[0]
    
    fire_prob   = probs[CLASS_NAMES.index('fire')].item()
    nofire_prob = probs[CLASS_NAMES.index('no_fire')].item()

    label = '🔥 YANGIN VAR' if fire_prob > nofire_prob else '✅ NORMAL'
    return {
        f'🔥 Yangın ({fire_prob:.1%})'   : fire_prob,
        f'✅ Normal ({nofire_prob:.1%})' : nofire_prob
    }, label

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type='pil', label='Uydu Görüntüsü Yükle'),
    outputs=[
        gr.Label(label='Olasılık Dağılımı'),
        gr.Textbox(label='Sonuç')
    ],
    title='🔥 Orman Yangını Erken Uyarı Sistemi',
    description='Uydu görüntüsü yükle → Yangın var mı yok mu öğren',
    examples=[],
    flagging_mode='never'
)

demo.launch(share=True)  # share=True → dışarıdan erişilebilir link

---
## 📡 Osman için API Notu

Osman'ın web sitesi modeli kullanmak istiyorsa `predict.py` dosyasını çalıştırıp:

```
POST /predict
Body: multipart/form-data  { file: <jpeg> }

Response: { "label": "fire" | "no_fire", "confidence": 0.95 }
```

şeklinde kullanabilir. `predict.py` dosyasına bakın.